# 2. Predict an isotopomer distribution from real data

Uses a model trained in `01_train.ipynb` to predict the isotopomer distribution of a measured metabolite, with uncertainty via Monte-Carlo dropout.

The pipeline is:

1. **Load** the measured HSQC + GC-MS data.
2. **Build feature vectors** in the same layout the model was trained on.
3. **Predict** the isotopomer distribution (mean + standard deviation).
4. **Back-simulate** HSQC/GC-MS from the prediction and save a results summary.

> One-command version: `mlp-predict --model saved_models/model_hsqc_0_1_1.keras --hsqc-vector 0 1 1 --hsqc-data hsqcData1.xlsx --gcms-data gcmsData1.xlsx --metabolite L-LacticAcid`

In [ ]:
import os
from metabolabpytools import isotopomerAnalysis

ia = isotopomerAnalysis.IsotopomerAnalysisNN()

hsqc_vector = [0, 1, 1]
n_carbons = len(hsqc_vector)
metabolite_name = 'L-LacticAcid'
model_path = 'saved_models/model_hsqc_0_1_1.keras'

## Step 1 — Load measured data

In [ ]:
hsqc_data_file = os.path.join(os.getcwd(), 'hsqcData1.xlsx')
gcms_data_file = os.path.join(os.getcwd(), 'gcmsData1.xlsx')
ia.load_hsqc_and_gcms_data(hsqc_data_file, gcms_data_file)
ia.inspect_metabolite_data(metabolite_name)

## Step 2 — Build feature vectors

Features are ordered/zero-padded against the full set of possible multiplets for this HSQC vector, matching the training layout exactly.

In [ ]:
X_real_data = ia.create_feature_vectors(metabolite_name, hsqc_vector)
X_real_data

## Step 3 — Predict

`n_iter` Monte-Carlo dropout passes give a mean prediction and a standard deviation (uncertainty). 200 is plenty; increase it for smoother uncertainty estimates.

In [ ]:
mean_predictions, std_dev_predictions, predicted_distributions = ia.load_model_and_predict(
    model_path, X_real_data, n_carbons, n_iter=200,
)

## Step 4 — Back-simulate and save results

Simulate the HSQC/GC-MS you'd expect from the predicted distribution, then write everything (real vs back-calculated) to `nn_analysis_results/`.

In [ ]:
predicted_hsqc_data, predicted_gcms_data = ia.simulate_from_predictions(predicted_distributions, hsqc_vector)

ia.save_results_summary(
    X_real_data, predicted_distributions, std_dev_predictions,
    predicted_hsqc_data, predicted_gcms_data, hsqc_vector,
)